# Rotate And Crop Video Batch

Batch-process concatenated behavior camera videos with per-video rotation and crop settings.

Workflow:
1. Build a per-video config table.
2. Preview rotation only for each video.
3. Edit any per-video angles or crop boxes.
4. Preview rotation plus crop for each video.
5. Run the full batch export.

In [ ]:
from pathlib import Path
import math
import ffmpeg
import pandas as pd
from IPython.display import Image, Markdown, display
from tqdm import tqdm


def probe_video(video_path):
    probe = ffmpeg.probe(str(video_path))
    video_stream = next(stream for stream in probe['streams'] if stream['codec_type'] == 'video')
    width = int(video_stream['width'])
    height = int(video_stream['height'])
    duration = float(video_stream['duration'])
    codec_name = video_stream.get('codec_name', 'Unknown')
    return {
        'width': width,
        'height': height,
        'duration_sec': duration,
        'codec_name': codec_name,
    }


def compute_rotated_canvas(width, height, angle_degrees):
    angle_rad = math.radians(angle_degrees)
    new_width = int(abs(width * math.cos(angle_rad)) + abs(height * math.sin(angle_rad)))
    new_height = int(abs(width * math.sin(angle_rad)) + abs(height * math.cos(angle_rad)))
    return angle_rad, new_width, new_height


def normalize_crop_box(crop_box=None, vertices=None):
    if crop_box is None and vertices is None:
        raise ValueError('Provide either crop_box or vertices.')

    if crop_box is not None:
        return {
            'crop_x': int(crop_box['crop_x']),
            'crop_y': int(crop_box['crop_y']),
            'crop_width': int(crop_box['crop_width']),
            'crop_height': int(crop_box['crop_height']),
        }

    x_coordinates = [v[0] for v in vertices]
    y_coordinates = [v[1] for v in vertices]
    x_min = min(x_coordinates)
    x_max = max(x_coordinates)
    y_min = min(y_coordinates)
    y_max = max(y_coordinates)
    return {
        'crop_x': int(x_min),
        'crop_y': int(y_min),
        'crop_width': int(x_max - x_min),
        'crop_height': int(y_max - y_min),
    }


def clamp_crop_box(crop_box, frame_width, frame_height):
    crop_x = max(0, int(crop_box['crop_x']))
    crop_y = max(0, int(crop_box['crop_y']))
    crop_width = int(crop_box['crop_width'])
    crop_height = int(crop_box['crop_height'])

    if crop_width <= 0 or crop_height <= 0:
        raise ValueError(f'Invalid crop size: {crop_width} x {crop_height}')

    if crop_x + crop_width > frame_width or crop_y + crop_height > frame_height:
        raise ValueError(
            f"Crop box {(crop_x, crop_y, crop_width, crop_height)} exceeds frame bounds {(frame_width, frame_height)}"
        )

    return {
        'crop_x': crop_x,
        'crop_y': crop_y,
        'crop_width': crop_width,
        'crop_height': crop_height,
    }


def build_output_path(input_path, output_dir, suffix='_rotated_cropped'):
    return output_dir / f'{input_path.stem}{suffix}.avi'


def build_video_config_df(input_dir, default_angle_degrees, default_crop_box, pattern='*.mp4'):
    input_dir = Path(input_dir)
    video_files = sorted(input_dir.glob(pattern))
    if not video_files:
        raise FileNotFoundError(f'No files matching {pattern!r} found in {input_dir}')

    rows = []
    for video_path in video_files:
        rows.append({
            'file_name': video_path.name,
            'input_file': str(video_path),
            'angle_degrees': float(default_angle_degrees),
            'crop_x': int(default_crop_box['crop_x']),
            'crop_y': int(default_crop_box['crop_y']),
            'crop_width': int(default_crop_box['crop_width']),
            'crop_height': int(default_crop_box['crop_height']),
            'enabled': True,
        })

    return pd.DataFrame(rows)


def build_rotated_stream(input_path, angle_degrees, preview_time_sec=None):
    info = probe_video(input_path)
    angle_rad, rotated_width, rotated_height = compute_rotated_canvas(
        info['width'], info['height'], angle_degrees
    )

    input_kwargs = {}
    if preview_time_sec is not None:
        input_kwargs['ss'] = max(0, float(preview_time_sec))

    stream = (
        ffmpeg
        .input(str(input_path), **input_kwargs)
        .filter(
            'pad',
            rotated_width,
            rotated_height,
            (rotated_width - info['width']) // 2,
            (rotated_height - info['height']) // 2,
            color='0xFFFFFF'
        )
        .filter('rotate', str(angle_rad))
    )

    transform_info = {
        'duration_sec': info['duration_sec'],
        'input_codec': info['codec_name'],
        'rotated_width': rotated_width,
        'rotated_height': rotated_height,
    }
    return stream, transform_info


def build_rotated_cropped_stream(input_path, angle_degrees, crop_box, preview_time_sec=None):
    stream, info = build_rotated_stream(
        input_path=input_path,
        angle_degrees=angle_degrees,
        preview_time_sec=preview_time_sec,
    )
    crop_box = clamp_crop_box(crop_box, info['rotated_width'], info['rotated_height'])
    stream = stream.filter(
        'crop',
        crop_box['crop_width'],
        crop_box['crop_height'],
        crop_box['crop_x'],
        crop_box['crop_y'],
    )
    info.update(crop_box)
    return stream, info


def render_frame_from_stream(stream, input_name):
    out, err = (
        stream
        .output('pipe:', vframes=1, format='image2', vcodec='png')
        .global_args('-loglevel', 'error')
        .run(capture_stdout=True, capture_stderr=True)
    )
    if not out:
        error_text = err.decode('utf-8', errors='replace').strip()
        raise RuntimeError(error_text or f'Could not render preview frame for {input_name}')
    return out


def preview_rotation_only(config_df, preview_time_fraction=0.5):
    results = []
    enabled_df = config_df.loc[config_df['enabled']].reset_index(drop=True)
    for row in tqdm(enabled_df.itertuples(index=False), total=len(enabled_df), desc='Rotation previews', unit='file'):
        input_path = Path(row.input_file)
        info = probe_video(input_path)
        preview_time_sec = info['duration_sec'] * preview_time_fraction
        stream, transform_info = build_rotated_stream(
            input_path=input_path,
            angle_degrees=row.angle_degrees,
            preview_time_sec=preview_time_sec,
        )
        image_bytes = render_frame_from_stream(stream, input_path.name)

        display(Markdown(
            f"### Rotation preview: {input_path.name}\n"
            f"Angle: `{row.angle_degrees}`  \n"
            f"Preview time: `{preview_time_sec:.2f}s`  \n"
            f"Rotated canvas: `{transform_info['rotated_width']} x {transform_info['rotated_height']}`"
        ))
        display(Image(data=image_bytes))

        results.append({
            'file_name': row.file_name,
            'preview_time_sec': preview_time_sec,
            'angle_degrees': row.angle_degrees,
            'rotated_width': transform_info['rotated_width'],
            'rotated_height': transform_info['rotated_height'],
        })

    return pd.DataFrame(results)


def preview_rotated_and_cropped(config_df, preview_time_fraction=0.5):
    results = []
    enabled_df = config_df.loc[config_df['enabled']].reset_index(drop=True)
    for row in tqdm(enabled_df.itertuples(index=False), total=len(enabled_df), desc='Crop previews', unit='file'):
        input_path = Path(row.input_file)
        info = probe_video(input_path)
        preview_time_sec = info['duration_sec'] * preview_time_fraction
        stream, transform_info = build_rotated_cropped_stream(
            input_path=input_path,
            angle_degrees=row.angle_degrees,
            crop_box={
                'crop_x': row.crop_x,
                'crop_y': row.crop_y,
                'crop_width': row.crop_width,
                'crop_height': row.crop_height,
            },
            preview_time_sec=preview_time_sec,
        )
        image_bytes = render_frame_from_stream(stream, input_path.name)

        display(Markdown(
            f"### Rotation + crop preview: {input_path.name}\n"
            f"Angle: `{row.angle_degrees}`  \n"
            f"Preview time: `{preview_time_sec:.2f}s`  \n"
            f"Crop box: `x={transform_info['crop_x']}, y={transform_info['crop_y']}, w={transform_info['crop_width']}, h={transform_info['crop_height']}`"
        ))
        display(Image(data=image_bytes))

        results.append({
            'file_name': row.file_name,
            'preview_time_sec': preview_time_sec,
            'angle_degrees': row.angle_degrees,
            'crop_x': transform_info['crop_x'],
            'crop_y': transform_info['crop_y'],
            'crop_width': transform_info['crop_width'],
            'crop_height': transform_info['crop_height'],
        })

    return pd.DataFrame(results)


def rotate_and_crop_video(input_path, output_path, angle_degrees, crop_box, overwrite=False):
    stream, info = build_rotated_cropped_stream(
        input_path=input_path,
        angle_degrees=angle_degrees,
        crop_box=crop_box,
    )

    ffmpeg_command = (
        stream
        .output(str(output_path), vcodec='mjpeg', qscale=3)
        .global_args('-progress', 'pipe:1', '-nostats', '-loglevel', 'error')
    )

    process = ffmpeg_command.run_async(
        pipe_stdout=True,
        pipe_stderr=True,
        overwrite_output=overwrite,
    )

    pbar = tqdm(
        total=info['duration_sec'],
        desc=input_path.name,
        unit='s',
        dynamic_ncols=True,
        leave=False,
    )

    for raw_line in process.stdout:
        line = raw_line.decode('utf-8', errors='replace').strip()
        if line.startswith('out_time_ms='):
            out_time_ms = int(line.split('=', 1)[1])
            current_time = out_time_ms / 1_000_000
            pbar.update(max(0, current_time - pbar.n))
        elif line == 'progress=end':
            break

    pbar.close()
    return_code = process.wait()
    stderr_output = process.stderr.read().decode('utf-8', errors='replace').strip()

    if return_code != 0:
        raise RuntimeError(stderr_output or f'ffmpeg failed with exit code {return_code}')

    return {
        'input_file': str(input_path),
        'output_file': str(output_path),
        'duration_sec': info['duration_sec'],
        'input_codec': info['input_codec'],
        'rotated_width': info['rotated_width'],
        'rotated_height': info['rotated_height'],
        'crop_x': info['crop_x'],
        'crop_y': info['crop_y'],
        'crop_width': info['crop_width'],
        'crop_height': info['crop_height'],
    }


def batch_rotate_and_crop_from_config(config_df, output_dir, overwrite=False):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    enabled_df = config_df.loc[config_df['enabled']].reset_index(drop=True)
    if enabled_df.empty:
        raise ValueError('No enabled videos found in config_df')

    results = []
    for row in tqdm(enabled_df.itertuples(index=False), total=len(enabled_df), desc='Files', unit='file'):
        input_path = Path(row.input_file)
        output_path = build_output_path(input_path, output_dir)

        if output_path.exists() and not overwrite:
            results.append({
                'input_file': str(input_path),
                'output_file': str(output_path),
                'status': 'skipped_existing',
            })
            continue

        try:
            result = rotate_and_crop_video(
                input_path=input_path,
                output_path=output_path,
                angle_degrees=row.angle_degrees,
                crop_box={
                    'crop_x': row.crop_x,
                    'crop_y': row.crop_y,
                    'crop_width': row.crop_width,
                    'crop_height': row.crop_height,
                },
                overwrite=overwrite,
            )
            result['status'] = 'processed'
            results.append(result)
        except Exception as exc:
            results.append({
                'input_file': str(input_path),
                'output_file': str(output_path),
                'status': 'error',
                'error': str(exc),
            })

    return pd.DataFrame(results)


In [ ]:
input_dir = Path('/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/YZ/Miniscope_data/Miniscope_data/Linear_track/BehavCamConcactenated_992')
output_dir = input_dir / 'rotated_and_cropped_avi'

default_angle_to_rotate = 30
default_crop_box = normalize_crop_box(vertices=[(51, 373), (709, 373), (709, 404), (51, 404)])
preview_time_fraction = 0.5
overwrite_existing = False

video_config_df = build_video_config_df(
    input_dir=input_dir,
    default_angle_degrees=default_angle_to_rotate,
    default_crop_box=default_crop_box,
    pattern='*.mp4',
)

print(f'Found {len(video_config_df)} input videos in {input_dir}')
print(f'Output directory: {output_dir}')
video_config_df

In [ ]:
# Edit per-video settings here before running previews or batch export.
# Examples:
# video_config_df.loc[video_config_df['file_name'] == 'm992_12302024_19_59_56_concactenatedbehavCam00_behavCam15.mp4', 'angle_degrees'] = 28
# video_config_df.loc[video_config_df['file_name'] == 'm992_12302024_19_59_56_concactenatedbehavCam00_behavCam15.mp4', ['crop_x', 'crop_y', 'crop_width', 'crop_height']] = [60, 370, 650, 34]
# video_config_df.loc[video_config_df['file_name'] == 'some_file.mp4', 'enabled'] = False

video_config_df

In [ ]:
rotation_preview_results = preview_rotation_only(
    config_df=video_config_df,
    preview_time_fraction=preview_time_fraction,
)

rotation_preview_results

In [ ]:
cropped_preview_results = preview_rotated_and_cropped(
    config_df=video_config_df,
    preview_time_fraction=preview_time_fraction,
)

cropped_preview_results

In [ ]:
batch_results = batch_rotate_and_crop_from_config(
    config_df=video_config_df,
    output_dir=output_dir,
    overwrite=overwrite_existing,
)

batch_results